<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%202%20-%20Knowledge%20with%20RAG/Learning/vector_embeddings_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Embeddings & Vector Databases (running on Gemini)

**Goal of this notebook:** understand, from first principles, how an LLM app turns text into numbers it can compare — and why we need a special kind of database to store and search those numbers.

We'll cover:
1. What is a vector embedding?
2. Similar meaning = similar vectors (cosine similarity)
3. Why a plain list doesn't scale — the need for a vector database
4. Chroma — a simple, beginner-friendly vector database
5. Putting it together: a mini RAG pipeline

> We use **Chroma** (instead of FAISS) because its API (`add`, `query`) reads almost like plain English, and it stores your original text + metadata alongside the vectors automatically. FAISS is faster for huge production indexes, but it only stores raw vectors — you'd have to manage the text mapping yourself, which adds noise while you're still learning the *concept*.

> ☁️ **This notebook calls Google's Gemini API in the cloud** — you'll need a free API key, but no local server to run. It works the same whether you run it locally (VS Code) or in Google Colab.

## 🔑 Step 0 — Get Your Gemini API Key

1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** and sign in with your Google account.
2. Click **Create API key** and copy it.
3. Store it as **`GAISTUDIO_API_KEY`**:
   - **VS Code / local**: open the `.env` file at the repo root and set `GAISTUDIO_API_KEY=your-key-here`
   - **Google Colab**: click the 🔑 key icon in the left sidebar → **Add new secret** → name it `GAISTUDIO_API_KEY` → paste your key → enable notebook access

Never paste your key directly into a code cell — the helper below reads it securely from whichever environment you're in.

In [1]:
%pip install -q google-genai chromadb numpy python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ Step 1 — Load the Key and Connect to Gemini

The `get_secret()` function below works the same whether you're in Colab or VS Code — it checks Colab's secret store first, then falls back to your local `.env` file.

In [3]:
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

print("✅ API key loaded successfully.")

✅ API key loaded successfully.


In [5]:
from google import genai

client = genai.Client(api_key=GAISTUDIO_API_KEY)

CHAT_MODEL = "gemini-3.6-flash"
EMBED_MODEL = "gemini-embedding-001"

print(f"✅ Connected. Chat model: {CHAT_MODEL} | Embedding model: {EMBED_MODEL}")

✅ Connected. Chat model: gemini-3.6-flash | Embedding model: gemini-embedding-001


## Part 1 — What is a vector embedding?

An **embedding** is just a list of numbers (a *vector*) that represents the *meaning* of a piece of text. A sentence about "dogs" and a sentence about "puppies" end up with vectors that point in roughly the same direction, even though they don't share many words.

Let's embed a single sentence — via the Gemini API — and look at what comes back.

In [6]:
response = client.models.embed_content(
    model=EMBED_MODEL,
    contents="The cat sat on the mat.",
)

vector = response.embeddings[0].values

print("How many numbers in the vector?", len(vector))
print("First 10 numbers:", vector[:10])

How many numbers in the vector? 3072
First 10 numbers: [-0.022607181, 0.015246871, 0.001684601, -0.0769077, 0.0045841224, 0.004410481, 0.002776919, 0.011506936, 0.005208321, 0.019448007]


Notice: thousands of numbers for one short sentence (Gemini's default embedding is much higher-dimensional than a typical local model), and none of them mean anything on their own — `-0.0123` isn't "cat" and `0.045` isn't "mat". The *meaning* only shows up when you compare this vector to another one.

That comparison is the whole point of embeddings, so let's do it next.

## Part 2 — Similar meaning = similar vectors

To compare two vectors we use **cosine similarity**: it measures the angle between them, ignoring their length.

- `1.0` → pointing in the exact same direction (identical meaning)
- `0.0` → unrelated (perpendicular)
- `-1.0` → opposite meaning

Let's embed a few sentences — two about the same topic, one completely different — and compare them.

In [7]:
import numpy as np

def embed(text):
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return np.array(result.embeddings[0].values)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [8]:
sentences = {
    "a": "The dog ran across the park.",
    "b": "A puppy sprinted through the garden.",
    "c": "The stock market fell sharply today.",
}

vectors = {key: embed(text) for key, text in sentences.items()}

print("a vs b (dog vs puppy)      :", round(float(cosine_similarity(vectors["a"], vectors["b"])), 4))
print("a vs c (dog vs stock market):", round(float(cosine_similarity(vectors["a"], vectors["c"])), 4))

a vs b (dog vs puppy)      : 0.7662
a vs c (dog vs stock market): 0.602


You should see the **dog/puppy** pair score noticeably higher than the **dog/stock market** pair — even though "dog" and "puppy" don't share any letters in common. The model has captured *meaning*, not just keywords.

This is the core trick behind semantic search: turn text into vectors, then find the vectors that are closest together.

## Part 3 — Why do we need a *vector database*?

Comparing 2 vectors with `cosine_similarity` is easy. But real apps have thousands or millions of documents. If you had to:

- embed every document once,
- keep all those vectors in memory,
- and loop over *all of them* every time a user asks a question...

...that's slow, and it doesn't scale. A **vector database** solves this by:

1. **Storing** vectors alongside their original text + metadata (so you don't have to manage that mapping yourself).
2. **Indexing** vectors so "find the closest ones" is fast, even with millions of entries.
3. **Persisting** everything to disk so you don't re-embed on every run.

Let's use **Chroma**, a lightweight vector database, to see this in action — still powered by Gemini for the embeddings.

## Part 4 — Chroma: a simple vector database

Chroma's API has three ideas to learn:

- `client.create_collection(...)` — like creating a table
- `collection.add(...)` — insert documents (Chroma embeds them for you)
- `collection.query(...)` — ask "which documents are closest to this text?"

We'll give Chroma a small custom embedding function that just calls our Gemini `embed()` logic, so the numbers stay consistent with what we just learned.

In [9]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        result = client.models.embed_content(model=EMBED_MODEL, contents=input)
        return [e.values for e in result.embeddings]

gemini_ef = GeminiEmbeddingFunction()

chroma_client = chromadb.Client()  # in-memory for this demo; use PersistentClient(path=...) to save to disk

collection = chroma_client.get_or_create_collection(
    name="bootcamp_demo",
    embedding_function=gemini_ef,
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_28664\3738679361.py:9: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  gemini_ef = GeminiEmbeddingFunction()


In [10]:
documents = [
    "The Eiffel Tower is located in Paris, France.",
    "Mount Everest is the tallest mountain above sea level.",
    "Python is a popular programming language for AI and data science.",
    "The Great Wall of China stretches over 13,000 miles.",
    "JavaScript is commonly used to build interactive websites.",
]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
)

print("Documents stored:", collection.count())

Documents stored: 5


In [11]:
results = collection.query(
    query_texts=["Which language should I learn for machine learning?"],
    n_results=2,
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"distance={distance:.4f}  ->  {doc}")

distance=0.6188  ->  Python is a popular programming language for AI and data science.
distance=0.9210  ->  JavaScript is commonly used to build interactive websites.


Chroma embedded our query (via Gemini), compared it against every stored vector, and returned the **closest matches** — the documents about programming languages, not the ones about mountains or landmarks. Under the hood it did the exact same cosine-similarity idea from Part 2, just indexed and automated for us.

## Part 5 — Putting it together: a mini RAG pipeline

**RAG (Retrieval-Augmented Generation)** = retrieve relevant chunks from a vector database, then hand them to an LLM as context so it can answer using *your* data instead of just what it memorized during training.

The flow:

```
question -> embed -> search vector DB -> top-k chunks -> stuff into prompt -> Gemini answer
```

Both the retrieval *and* the generation step below run through the Gemini API.

In [12]:
def retrieve(query: str, k: int = 2) -> list[str]:
    results = collection.query(query_texts=[query], n_results=k)
    return results["documents"][0]

def ask(query: str) -> str:
    context = "\n".join(retrieve(query))
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer using only the context above."
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=prompt,
    )
    return response.text

print(ask("What is the tallest mountain?"))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the context provided, Mount Everest is the tallest mountain (above sea level).


## Recap

- An **embedding** turns text into a vector of numbers that captures meaning.
- **Cosine similarity** compares two vectors to see how related they are.
- A **vector database** (like Chroma) stores many vectors + their original text, and makes "find the closest ones" fast at scale — that's what a plain Python list can't do well.
- **RAG** = retrieve the closest chunks from a vector database, then let the LLM answer using that context.
- **Gemini** gave us both an embedding model (`gemini-embedding-001`) and a chat model (`gemini-3.6-flash`) through the same `google-genai` client — no local server required, and it runs the same in VS Code or Colab.

Next: try swapping in your own documents in Part 4, comparing these results with the LM Studio version (`vector_embeddings.ipynb`), or explore `PersistentClient` so your Chroma collection survives across notebook restarts.